In [268]:
#!pip install lbforaging

In [269]:
import lbforaging
from games.foraging import Foraging 
import numpy as np
import time
from agents.random_agent import RandomAgent
from agents.jal_am import JointActionLearningAM

In [270]:
game = Foraging(config=None, seed=1)

In [271]:
game.reset()
for agent in game.agents:
    print(f"Agent {agent}")
    print(f"Observed state: {game.observe(agent)}")
    print(f"Observed actions: {game.observations[agent]}")

Agent agent_0
Observed state: [6. 2. 2. 4. 6. 2. 0. 1. 2.]
Observed actions: [6. 2. 2. 4. 6. 2. 0. 1. 2.]
Agent agent_1
Observed state: [6. 2. 2. 0. 1. 2. 4. 6. 2.]
Observed actions: [6. 2. 2. 0. 1. 2. 4. 6. 2.]


In [272]:
def play_episode(game, agents, verbose=False, render=False):

    # Initialize the game
    game.reset()

    # Print initial observations
    if verbose:
        print(f"Step: {game.current_step}")
        for agent in game.agents:
            print(f"Agent {agent}: {game.observe(agent)}")

    # Initialize rewards for each agent
    cum_rewards = dict(map(lambda agent: (agent, 0), game.agents))

    # render the game if required
    if render:
        game.render()
        time.sleep(0.5)

    while not game.done():

        # Get actions from each agent
        actions = dict(map(lambda agent: (agent, agents[agent].action()), game.agents))

        # Perform the actions in the game
        game.step(actions)

        # Update the cum_rewards for each agent
        for agent in game.agents:
            cum_rewards[agent] += game.reward(agent)

        # Print the rewards if verbose is enabled
        if verbose:
            print(f"Step: {game.current_step}")
            for agent in game.agents:
                print(f"Agent {agent} action: {actions[agent]} ({game.action_set[actions[agent]]})")
                print(f"Agent {agent} reward: {game.reward(agent)}")
                print(f"Agent {agent} next state: {game.observe(agent)}")
                print(f"Agent {agent} joint action: {game.observations[agent]}")
        
        if render:
            game.render()
            time.sleep(0.5)
        
    return cum_rewards

In [273]:
agent_dict = dict(map(lambda agent: (agent, RandomAgent(game=game, agent=agent)), game.agents))

In [274]:
def run(game, agent_dict, n_episodes=100, verbose=False, render=False):
    total_rewards = dict(map(lambda agent: (agent, 0), game.agents))
    for episode in range(n_episodes):
        if verbose:
            print(f"-- Episode: {episode}")
        cum_rewards = play_episode(game, agent_dict, verbose=verbose, render=render)
        for agent in game.agents:
            total_rewards[agent] += cum_rewards[agent]
        if verbose:
            for agent in game.agents:
                print(f"Total rewards {agent}: {total_rewards[agent]}")
            print("--")
    return total_rewards

In [275]:
run(game=game, agent_dict=agent_dict, n_episodes=1, verbose=True, render=False)

-- Episode: 0
Step: 0
Agent agent_0: [2. 6. 1. 7. 0. 1. 6. 7. 1.]
Agent agent_1: [2. 6. 1. 6. 7. 1. 7. 0. 1.]
Step: 1
Agent agent_0 action: 4 (EAST)
Agent agent_0 reward: 0
Agent agent_0 next state: [2. 6. 1. 7. 1. 1. 6. 7. 1.]
Agent agent_0 joint action: [2. 6. 1. 7. 1. 1. 6. 7. 1.]
Agent agent_1 action: 4 (EAST)
Agent agent_1 reward: 0
Agent agent_1 next state: [2. 6. 1. 6. 7. 1. 7. 1. 1.]
Agent agent_1 joint action: [2. 6. 1. 6. 7. 1. 7. 1. 1.]
Step: 2
Agent agent_0 action: 3 (WEST)
Agent agent_0 reward: 0
Agent agent_0 next state: [2. 6. 1. 7. 0. 1. 6. 7. 1.]
Agent agent_0 joint action: [2. 6. 1. 7. 0. 1. 6. 7. 1.]
Agent agent_1 action: 5 (LOAD)
Agent agent_1 reward: 0
Agent agent_1 next state: [2. 6. 1. 6. 7. 1. 7. 0. 1.]
Agent agent_1 joint action: [2. 6. 1. 6. 7. 1. 7. 0. 1.]
Step: 3
Agent agent_0 action: 1 (NORTH)
Agent agent_0 reward: 0
Agent agent_0 next state: [2. 6. 1. 6. 0. 1. 6. 7. 1.]
Agent agent_0 joint action: [2. 6. 1. 6. 0. 1. 6. 7. 1.]
Agent agent_1 action: 5 (LOAD)

{'agent_0': 0, 'agent_1': 0}

In [276]:
jalam_agent_dict = dict(map(lambda agent: (agent, JointActionLearningAM(game=game, agent=agent)), game.agents))

In [277]:
run(game=game, agent_dict=jalam_agent_dict, n_episodes=1, verbose=True, render=False)

-- Episode: 0
Step: 0
Agent agent_0: [6. 2. 2. 4. 6. 2. 0. 1. 2.]
Agent agent_1: [6. 2. 2. 0. 1. 2. 4. 6. 2.]
Step: 1
Agent agent_0 action: 3 (WEST)
Agent agent_0 reward: 0
Agent agent_0 next state: [6. 2. 2. 4. 5. 2. 0. 1. 2.]
Agent agent_0 joint action: [6. 2. 2. 4. 5. 2. 0. 1. 2.]
Agent agent_1 action: 5 (LOAD)
Agent agent_1 reward: 0
Agent agent_1 next state: [6. 2. 2. 0. 1. 2. 4. 5. 2.]
Agent agent_1 joint action: [6. 2. 2. 0. 1. 2. 4. 5. 2.]
Step: 2
Agent agent_0 action: 4 (EAST)
Agent agent_0 reward: 0
Agent agent_0 next state: [6. 2. 2. 4. 6. 2. 1. 1. 2.]
Agent agent_0 joint action: [6. 2. 2. 4. 6. 2. 1. 1. 2.]
Agent agent_1 action: 2 (SOUTH)
Agent agent_1 reward: 0
Agent agent_1 next state: [6. 2. 2. 1. 1. 2. 4. 6. 2.]
Agent agent_1 joint action: [6. 2. 2. 1. 1. 2. 4. 6. 2.]
Step: 3
Agent agent_0 action: 4 (EAST)
Agent agent_0 reward: 0
Agent agent_0 next state: [6. 2. 2. 4. 7. 2. 1. 1. 2.]
Agent agent_0 joint action: [6. 2. 2. 4. 7. 2. 1. 1. 2.]
Agent agent_1 action: 0 (NONE)

{'agent_0': 0, 'agent_1': 0}